# 머신러닝 기초 3장. 부스팅

사회약학 연구방법 노트 '머신러닝 기초' 3장을 따라 하는 노트북입니다. 0장과 같은 가상 청구자료와 분할로 그레이디언트 부스팅의 원리를 손으로 따라가고, XGBoost, LightGBM, 히스토그램 부스팅, CatBoost를 만들어 조기 종료와 Optuna로 튜닝한 뒤 SHAP 값으로 결과를 정리합니다. 셀을 위에서부터 차례로 실행하세요. 튜닝 셀은 Colab에서 몇 분씩 걸립니다. 자료는 사이트가 만든 가상 자료이며 실제 환자 자료가 아닙니다.


## 설치

Colab에서는 세션마다 이 셀부터 실행합니다. LightGBM과 shap은 Colab에 이미 있을 수 있고, 그러면 `Requirement already satisfied`로 나옵니다. 설치 뒤 세션을 다시 시작하라는 안내가 나오면 따르고 셀 2부터 다시 실행합니다.

### 셀 1. Optuna, HyperOpt, LightGBM, CatBoost, shap 설치 (Colab은 세션마다)

In [ ]:
!pip install optuna hyperopt lightgbm catboost shap

## 준비

0장의 자료 불러오기와 분할, 0–1장의 전처리 함수, 이 장에서 새로 쓰는 `make_nan_pipe`(숫자 열의 결측을 그대로 두는 전처리), 1장의 `boot_auc`를 한 셀에 모았습니다.

### 셀 2. 0–2장의 준비를 한 번에 (자료, 분할, 파이프라인 세 가지, skf)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, cross_validate)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, log_loss

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
df = pd.read_csv(BASE + "ml_claims.csv", storage_options=UA)
y = df["admit_2023"]
X = df.drop(columns=["id", "admit_2023", "cost_2023"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=2026)
cat_cols = ["insurance", "region", "smoking", "alcohol"]
num_cols = [c for c in X.columns if c not in cat_cols]

def cat_steps():
    """글자 열: 결측을 'missing' 범주로 채운 뒤 원-핫 (0-2장과 같음)"""
    return Pipeline([
        ("impute", SimpleImputer(strategy="constant",
                                 fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore",
                                 sparse_output=False))])

def make_pipe(model, num=num_cols, cat=cat_cols):
    """0장: 중앙값 대체 + 표준화 + 원-핫"""
    num_steps = Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())])
    prep = ColumnTransformer([("num", num_steps, num),
                              ("cat", cat_steps(), cat)])
    return Pipeline([("prep", prep), ("model", model)])

def make_tree_pipe(model, num=num_cols, cat=cat_cols):
    """1장: 중앙값 대체 + 원-핫 (표준화 없음)"""
    prep = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), num),
         ("cat", cat_steps(), cat)],
        verbose_feature_names_out=False)
    return Pipeline([("prep", prep), ("model", model)])

def make_nan_pipe(model, num=num_cols, cat=cat_cols):
    """3장: 숫자 열은 결측(NaN) 그대로, 글자 열만 원-핫"""
    prep = ColumnTransformer(
        [("num", "passthrough", num), ("cat", cat_steps(), cat)],
        verbose_feature_names_out=False).set_output(transform="pandas")
    return Pipeline([("prep", prep), ("model", model)])

def boot_auc(y_true, p, n_boot=1000, seed=2026):
    """1장 마 절: 시험 자료를 복원추출해 다시 구한 AUC n_boot개"""
    rng = np.random.default_rng(seed)
    yt, p, out = np.asarray(y_true), np.asarray(p), []
    for _ in range(n_boot):
        i = rng.integers(0, len(yt), len(yt))
        out.append(roc_auc_score(yt[i], p[i]))
    return np.array(out)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
print(X_train.shape, X_test.shape, y_train.sum(), y_test.sum())

## 가. 앞 나무의 오차를 다음 나무가 배운다

### 셀 3. 환자 8명으로 그레이디언트 부스팅 두 번 돌리기

In [ ]:
from sklearn.tree import DecisionTreeRegressor

toy = pd.DataFrame({
    "age":   [52, 61, 47, 78, 83, 69, 88, 74],
    "n_ed":  [0, 0, 1, 2, 0, 2, 1, 1],
    "admit": [0, 0, 0, 1, 0, 1, 1, 0]}, index=range(1, 9))
lr = 0.5                                   # 학습률 (보기 쉽게 크게)
p = np.full(8, toy["admit"].mean())        # ① 모두 같은 출발 확률
for r in [1, 2]:
    resid = toy["admit"] - p               # ② 남은 오차
    stump = DecisionTreeRegressor(max_depth=1, random_state=2026)
    stump.fit(toy[["age", "n_ed"]], resid) # ③ 작은 나무가 오차를 배움
    step = stump.predict(toy[["age", "n_ed"]])
    p = p + lr * step                      # ④ 학습률만큼만 고침
    f = ["age", "n_ed"][stump.tree_.feature[0]]
    print(f"round {r}: {f} <= {stump.tree_.threshold[0]}")
    toy[f"resid{r}"], toy[f"tree{r}"] = resid, step
    toy[f"p{r}"] = p
toy.round(3)

### 셀 4. 나무 수에 따른 학습·검증 손실 (staged_predict_proba)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

X_tr, X_val, y_tr, y_val = train_test_split(   # 학습 자료를 다시 나눔
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=2026)
prep = make_tree_pipe(None).named_steps["prep"]
A_tr = prep.fit_transform(X_tr)                 # 9000 × 48
A_val = prep.transform(X_val)                   # 2250 × 48

gbm = GradientBoostingClassifier(n_estimators=500, learning_rate=0.1,
                                 max_depth=3, random_state=2026)
gbm.fit(A_tr, y_tr)
loss_tr = [log_loss(y_tr, p[:, 1])
           for p in gbm.staged_predict_proba(A_tr)]
loss_val = [log_loss(y_val, p[:, 1])
            for p in gbm.staged_predict_proba(A_val)]
best = int(np.argmin(loss_val)) + 1             # 검증 손실이 가장 작은 나무 수
for k in [1, 50, best, 500]:
    print(f"{k:3d} trees: train {loss_tr[k - 1]:.4f},"
          f" validation {loss_val[k - 1]:.4f}")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(range(1, 501), loss_tr, label="training (9,000)")
ax.plot(range(1, 501), loss_val, label="validation (2,250)")
ax.axvline(best, ls="--", color="gray")
ax.set_xlabel("Number of trees")
ax.set_ylabel("Log loss")
ax.legend()
plt.tight_layout()

### 셀 5. 사이킷런 그레이디언트 부스팅의 기본값으로 5겹 교차검증

In [ ]:
t0 = time.perf_counter()
cv_gbm = cross_val_score(
    make_tree_pipe(GradientBoostingClassifier(random_state=2026)),
    X_train, y_train, cv=skf, scoring="roc_auc")
t_gbm = time.perf_counter() - t0
print(f"CV AUC {cv_gbm.mean():.4f} (SD {cv_gbm.std():.4f}),"
      f" {t_gbm:.0f} s")

## 나. XGBoost

### 셀 6. XGBoost를 파이프라인 안에서: 결측 대체와 NaN 그대로

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier
print("xgboost", xgb.__version__)

shallow = dict(max_depth=2, learning_rate=0.05, n_estimators=300)
rows = []
for how, maker in [("median imputed", make_tree_pipe),
                   ("NaN as is", make_nan_pipe)]:
    for name, params in [("default", {}), ("shallow", shallow)]:
        model = maker(XGBClassifier(random_state=2026, **params))
        auc = cross_val_score(model, X_train, y_train, cv=skf,
                              scoring="roc_auc")
        rows.append([how, name, auc.mean(), auc.std()])
pd.DataFrame(rows, columns=["missing", "setting", "CV AUC", "SD"]
             ).round(4)

### 셀 7. XGBoost 안을 들여다보기: 실제 기본값과 결측의 방향

In [ ]:
import json

prep_nan = make_nan_pipe(None).named_steps["prep"]
Z_train = prep_nan.fit_transform(X_train)   # DataFrame, NaN 그대로
Z_test = prep_nan.transform(X_test)
small = XGBClassifier(n_estimators=50, max_depth=2, learning_rate=0.1,
                      random_state=2026).fit(Z_train, y_train)

cfg = json.loads(small.get_booster().save_config())   # 실제로 쓴 설정
tp = cfg["learner"]["gradient_booster"]["tree_train_param"]
print({k: tp[k] for k in ["eta", "max_depth", "min_child_weight",
                          "lambda", "alpha", "gamma", "max_bin"]})
trees = small.get_booster().trees_to_dataframe()
cols = ["Tree", "ID", "Feature", "Split", "Yes", "No", "Missing", "Gain"]
print(trees.loc[trees["Tree"] == 0, cols].to_string(index=False))
bmi = trees[trees["Feature"] == "bmi"]
print("bmi splits:", len(bmi), " NaN sent to the 'Yes' side:",
      (bmi["Missing"] == bmi["Yes"]).sum())

### 셀 8. max_depth의 검증 곡선

In [ ]:
from sklearn.model_selection import validation_curve

depths = [1, 2, 3, 4, 5, 6, 8]
tr, va = validation_curve(
    make_nan_pipe(XGBClassifier(learning_rate=0.05, n_estimators=300,
                                random_state=2026)),
    X_train, y_train, param_name="model__max_depth",
    param_range=depths, cv=skf, scoring="roc_auc")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for s, lab in [(tr, "training folds"), (va, "cross-validation")]:
    m, sd = s.mean(axis=1), s.std(axis=1)
    ax.plot(depths, m, "o-", label=lab)
    ax.fill_between(depths, m - sd, m + sd, alpha=0.2)
ax.set_xlabel("max_depth (learning_rate 0.05, 300 trees)")
ax.set_ylabel("AUC")
ax.legend()
plt.tight_layout()
pd.DataFrame({"max_depth": depths, "train AUC": tr.mean(axis=1),
              "CV AUC": va.mean(axis=1), "CV SD": va.std(axis=1)}).round(4)

## 다. LightGBM

이 절의 셀은 LightGBM이 필요합니다(Colab에는 있습니다). 사이트의 출력은 Colab에서 실행한 결과입니다.

### 셀 9. LightGBM 기본값과 잎을 줄인 설정

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
print("lightgbm", lgb.__version__)

few = dict(num_leaves=4, learning_rate=0.05, n_estimators=300,
           min_child_samples=50)
for name, params in [("default", {}), ("4 leaves", few)]:
    model = make_nan_pipe(LGBMClassifier(random_state=2026, verbose=-1,
                                         **params))
    t0 = time.perf_counter()
    auc = cross_val_score(model, X_train, y_train, cv=skf,
                          scoring="roc_auc")
    print(f"{name:9s} CV AUC {auc.mean():.4f} (SD {auc.std():.4f}),"
          f" {time.perf_counter() - t0:.1f} s")

### 셀 10. num_leaves의 검증 곡선

In [ ]:
leaves = [2, 4, 8, 16, 31, 64]
rows = []
for nl in leaves:
    model = make_nan_pipe(LGBMClassifier(
        num_leaves=nl, learning_rate=0.05, n_estimators=300,
        random_state=2026, verbose=-1))
    r = cross_validate(model, X_train, y_train, cv=skf,
                       scoring="roc_auc", return_train_score=True)
    rows.append([nl, r["train_score"].mean(), r["test_score"].mean(),
                 r["test_score"].std()])
res_leaves = pd.DataFrame(rows, columns=["num_leaves", "train AUC",
                                         "CV AUC", "CV SD"])
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(leaves, res_leaves["train AUC"], "o-", label="training folds")
ax.plot(leaves, res_leaves["CV AUC"], "o-", label="cross-validation")
ax.set_xscale("log", base=2)
ax.set_xlabel("num_leaves (learning_rate 0.05, 300 trees)")
ax.set_ylabel("AUC")
ax.legend()
plt.tight_layout()
res_leaves.round(4)

### 셀 11. 범주형 열을 그대로 넘기기 (pandas category)

In [ ]:
Xl_train = X_train.astype({c: "category" for c in cat_cols})
print([str(t) for t in Xl_train[cat_cols].dtypes])
for name, model, data in [
        ("one-hot (make_nan_pipe)",
         make_nan_pipe(LGBMClassifier(random_state=2026, verbose=-1,
                                      **few)), X_train),
        ("category dtype, no pipeline",
         LGBMClassifier(random_state=2026, verbose=-1, **few), Xl_train)]:
    auc = cross_val_score(model, data, y_train, cv=skf,
                          scoring="roc_auc")
    print(f"{name:28s} CV AUC {auc.mean():.4f}")

## 라. 히스토그램 부스팅과 CatBoost

### 셀 12. 히스토그램 부스팅 (HistGradientBoostingClassifier)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

def make_hgb_pipe(model):
    """글자 열은 정수 번호로(결측은 NaN 그대로) 앞에, 숫자 열은 그대로"""
    prep = ColumnTransformer(
        [("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                                unknown_value=np.nan), cat_cols)],
        remainder="passthrough", verbose_feature_names_out=False)
    return Pipeline([("prep", prep), ("model", model)])

settings = {
    "default": {},
    "early_stopping=True": dict(early_stopping=True),
    "tuned": dict(learning_rate=0.05, max_leaf_nodes=4,
                  min_samples_leaf=50, max_iter=1000,
                  early_stopping=True, n_iter_no_change=30)}
rows = []
for name, params in settings.items():
    hgb = make_hgb_pipe(HistGradientBoostingClassifier(
        categorical_features=[0, 1, 2, 3], random_state=2026, **params))
    auc = cross_val_score(hgb, X_train, y_train, cv=skf,
                          scoring="roc_auc")
    hgb.fit(X_train, y_train)                 # 11,250명 전체로
    rows.append([name, auc.mean(), auc.std(), hgb[-1].n_iter_])
pd.DataFrame(rows, columns=["setting", "CV AUC", "SD",
                            "trees (all 11,250)"]).round(4)

### 셀 13. CatBoost: 범주형 열을 그대로, 기본값으로

In [ ]:
import catboost
from catboost import CatBoostClassifier
print("catboost", catboost.__version__)

Xk_train, Xk_test = X_train.copy(), X_test.copy()
Xk_train[cat_cols] = Xk_train[cat_cols].fillna("missing")  # 글자 열의 NaN은
Xk_test[cat_cols] = Xk_test[cat_cols].fillna("missing")    # 글자로 바꿔야 함

def cv_catboost(**params):
    """CatBoost의 5겹 교차검증 AUC (겹은 skf와 같음)"""
    aucs = []
    for tr, va in skf.split(Xk_train, y_train):
        m = CatBoostClassifier(cat_features=cat_cols, random_seed=2026,
                               verbose=0, **params)
        m.fit(Xk_train.iloc[tr], y_train.iloc[tr])
        p = m.predict_proba(Xk_train.iloc[va])[:, 1]
        aucs.append(roc_auc_score(y_train.iloc[va], p))
    return np.mean(aucs), np.std(aucs), m

for name, params in [("default", {}),
                     ("ordered", {"boosting_type": "Ordered"})]:
    t0 = time.perf_counter()
    mean, sd, m = cv_catboost(**params)
    if name == "default":
        cv_cat_default = mean
    used = m.get_all_params()
    print(f"{name:8s} CV AUC {mean:.4f} (SD {sd:.4f}),"
          f" {time.perf_counter() - t0:.0f} s")
    print("   ", {k: used[k] for k in ["boosting_type", "iterations",
                                       "learning_rate", "depth",
                                       "l2_leaf_reg"]})

## 마. 조기 종료와 튜닝

### 셀 14. XGBoost의 조기 종료 (eval_set, early_stopping_rounds, evals_result)

In [ ]:
Z_tr, Z_val = Z_train.loc[X_tr.index], Z_train.loc[X_val.index]
es_model = XGBClassifier(n_estimators=2000, learning_rate=0.05,
                         max_depth=2, early_stopping_rounds=50,
                         eval_metric="logloss", random_state=2026)
es_model.fit(Z_tr, y_tr, eval_set=[(Z_tr, y_tr), (Z_val, y_val)],
             verbose=False)
hist = es_model.evals_result()
print(list(hist), " trees grown:", len(hist["validation_1"]["logloss"]))
print("best_iteration:", es_model.best_iteration,
      " best validation log loss:", round(es_model.best_score, 4))
p_val = es_model.predict_proba(Z_val)[:, 1]   # 가장 좋았던 지점까지의 나무로
print("validation AUC:", round(roc_auc_score(y_val, p_val), 4))

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(hist["validation_0"]["logloss"], label="training (9,000)")
ax.plot(hist["validation_1"]["logloss"], label="validation (2,250)")
ax.axvline(es_model.best_iteration, ls="--", color="gray")
ax.set_xlabel("Boosting round (tree number - 1)")
ax.set_ylabel("Log loss")
ax.legend()
plt.tight_layout()

### 셀 15. LightGBM의 조기 종료 (callbacks)

In [ ]:
lgb_es = LGBMClassifier(n_estimators=2000, learning_rate=0.05,
                        num_leaves=4, random_state=2026, verbose=-1)
lgb_es.fit(Z_tr, y_tr, eval_set=[(Z_tr, y_tr), (Z_val, y_val)],
           eval_metric="binary_logloss",
           callbacks=[lgb.early_stopping(50, verbose=False),
                      lgb.log_evaluation(0)])
print("best_iteration_:", lgb_es.best_iteration_)
print({k: round(v["binary_logloss"], 4)
       for k, v in lgb_es.best_score_.items()})
p_lval = lgb_es.predict_proba(Z_val)[:, 1]
print("validation AUC:", round(roc_auc_score(y_val, p_lval), 4))
ax = lgb.plot_metric(lgb_es, metric="binary_logloss", figsize=(6.5, 3.4))

### 셀 16. 교차검증 안의 조기 종료: 검증 겹을 두 번 쓰면

In [ ]:
def fit_es(Z_a, y_a, Z_b, y_b, metric):
    """Z_b로 멈출 곳을 정하는 XGBoost (깊이 2, 학습률 0.05)"""
    m = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=2,
                      early_stopping_rounds=50, eval_metric=metric,
                      random_state=2026)
    return m.fit(Z_a, y_a, eval_set=[(Z_b, y_b)], verbose=False)

for metric in ["logloss", "auc"]:
    leak, clean = [], []
    for tr, va in skf.split(Z_train, y_train):
        Zb, yb = Z_train.iloc[va], y_train.iloc[va]   # 검증 겹
        Za, Zs, ya, ys = train_test_split(            # 학습 겹을 8 대 2로
            Z_train.iloc[tr], y_train.iloc[tr], test_size=0.2,
            stratify=y_train.iloc[tr], random_state=2026)
        m = fit_es(Za, ya, Zb, yb, metric)            # 검증 겹으로 멈춤
        leak.append(roc_auc_score(yb, m.predict_proba(Zb)[:, 1]))
        m = fit_es(Za, ya, Zs, ys, metric)            # 떼어 둔 20%로 멈춤
        clean.append(roc_auc_score(yb, m.predict_proba(Zb)[:, 1]))
    d = np.array(leak) - np.array(clean)
    print(f"stop on {metric:7s}: leaky {np.mean(leak):.4f},"
          f" clean {np.mean(clean):.4f}, leaky higher in"
          f" {(d > 0).sum()} of 5 folds")

### 셀 17. 튜닝 순서의 실전 요령을 격자 탐색으로

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {"learning_rate": 0.1, "n_estimators": 200}   # 학습률을 크게
steps = [{"max_depth": [1, 2, 3, 4],                  # 1 나무 구조
          "min_child_weight": [1, 5, 20]},
         {"subsample": [0.6, 0.8, 1.0],               # 2 표본·특성 비율
          "colsample_bytree": [0.4, 0.7, 1.0]},
         {"reg_lambda": [0.1, 1, 10, 50]}]            # 3 규제
t0 = time.perf_counter()
for k, grid in enumerate(steps, start=1):
    gs = GridSearchCV(
        make_nan_pipe(XGBClassifier(random_state=2026, **params)),
        {"model__" + name: v for name, v in grid.items()},
        cv=skf, scoring="roc_auc")
    gs.fit(X_train, y_train)
    params.update({name.replace("model__", ""): v
                   for name, v in gs.best_params_.items()})
    print(f"step {k}: CV AUC {gs.best_score_:.4f}", gs.best_params_)
params.update(learning_rate=0.02, n_estimators=1000)  # 4 학습률 ÷ 5, 나무 × 5
cv4 = cross_val_score(make_nan_pipe(XGBClassifier(random_state=2026,
                                                  **params)),
                      X_train, y_train, cv=skf, scoring="roc_auc")
t_steps = time.perf_counter() - t0
print(f"step 4: CV AUC {cv4.mean():.4f}  ({t_steps:.0f} s)")
step_params = params

### 셀 18. Optuna로 XGBoost 튜닝 (60번 시도)

In [ ]:
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    """조합 하나를 골라 5겹 교차검증 AUC를 돌려준다 (겹은 늘 skf)"""
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 1000,
                                       step=50),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3,
                                          log=True),
        max_depth=trial.suggest_int("max_depth", 1, 6),
        min_child_weight=trial.suggest_float("min_child_weight", 0.5,
                                             50, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3,
                                             1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.1, 100,
                                       log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 0.001, 10,
                                      log=True))
    model = make_nan_pipe(XGBClassifier(random_state=2026, **params))
    return cross_val_score(model, X_train, y_train, cv=skf,
                           scoring="roc_auc").mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize",
                            sampler=TPESampler(seed=2026))
t0 = time.perf_counter()
study.optimize(objective, n_trials=60)
t_optuna = time.perf_counter() - t0
best_xgb = study.best_params
print({k: round(v, 4) for k, v in best_xgb.items()})
print(round(study.best_value, 4), " trial", study.best_trial.number,
      f" ({t_optuna:.0f} s)")

### 셀 19. Optuna 결과를 그림과 표로

In [ ]:
from optuna.visualization.matplotlib import (
    plot_optimization_history, plot_param_importances, plot_slice)
from optuna.importance import FanovaImportanceEvaluator

ax1 = plot_optimization_history(study)
ax2 = plot_param_importances(
    study, evaluator=FanovaImportanceEvaluator(seed=2026))
ax3 = plot_slice(study, params=["learning_rate", "n_estimators",
                                "max_depth"])
tdf = study.trials_dataframe(attrs=("number", "value", "params"))
tdf.columns = [c.replace("params_", "") for c in tdf.columns]
tdf.sort_values("value", ascending=False).head(5).round(3)

### 셀 20. HyperOpt로 같은 일 (교재의 방식, 30번)

In [ ]:
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

hp_space = {
    "max_depth": hp.quniform("max_depth", 1, 6, 1),
    "n_estimators": hp.quniform("n_estimators", 100, 1000, 50),
    "learning_rate": hp.loguniform("learning_rate",
                                   np.log(0.01), np.log(0.3)),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.3, 1.0)}

def hp_objective(params):
    model = make_nan_pipe(XGBClassifier(
        random_state=2026, max_depth=int(params["max_depth"]),
        n_estimators=int(params["n_estimators"]),
        learning_rate=params["learning_rate"],
        colsample_bytree=params["colsample_bytree"]))
    auc = cross_val_score(model, X_train, y_train, cv=skf,
                          scoring="roc_auc").mean()
    return {"loss": -auc, "status": STATUS_OK}

trials = Trials()
t0 = time.perf_counter()
best_hp = fmin(hp_objective, hp_space, algo=tpe.suggest, max_evals=30,
               trials=trials, rstate=np.random.default_rng(2026),
               show_progressbar=False)
t_hyperopt = time.perf_counter() - t0
print({k: round(float(v), 4) for k, v in best_hp.items()})
print(round(-min(trials.losses()), 4), f" ({t_hyperopt:.0f} s)")

### 셀 21. 기본값, 단계별 격자, Optuna의 교차검증과 시험 AUC

In [ ]:
cands = {"default": {},
         "stepwise grid": step_params,
         "Optuna": best_xgb}
rows, fitted = [], {}
for name, params in cands.items():
    model = make_nan_pipe(XGBClassifier(random_state=2026, **params))
    cv = cross_val_score(model, X_train, y_train, cv=skf,
                         scoring="roc_auc")
    fitted[name] = model.fit(X_train, y_train)    # 학습 자료 전체로
    p = fitted[name].predict_proba(X_test)[:, 1]
    lo, hi = np.percentile(boot_auc(y_test, p), [2.5, 97.5])
    rows.append([name, cv.mean(), cv.std(), roc_auc_score(y_test, p),
                 lo, hi])
xgb_final = fitted["Optuna"]                       # 교차검증으로 고른 모형
p_xgb = xgb_final.predict_proba(X_test)[:, 1]
pd.DataFrame(rows, columns=["XGBoost", "CV AUC", "CV SD", "test AUC",
                            "CI low", "CI high"]).round(3)

### 셀 22. Optuna로 LightGBM 튜닝 (40번 시도)

In [ ]:
def lgb_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 1000,
                                       step=50),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3,
                                          log=True),
        num_leaves=trial.suggest_int("num_leaves", 2, 64, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200,
                                            log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3,
                                             1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.01, 100,
                                       log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 0.001, 10,
                                      log=True))
    model = make_nan_pipe(LGBMClassifier(
        subsample_freq=1, random_state=2026, verbose=-1, **params))
    return cross_val_score(model, X_train, y_train, cv=skf,
                           scoring="roc_auc").mean()

study_lgb = optuna.create_study(direction="maximize",
                                sampler=TPESampler(seed=2026))
t0 = time.perf_counter()
study_lgb.optimize(lgb_objective, n_trials=40)
print({k: round(v, 4) for k, v in study_lgb.best_params.items()})
print(round(study_lgb.best_value, 4), " trial",
      study_lgb.best_trial.number,
      f" ({time.perf_counter() - t0:.0f} s)")

## 바. 변수 기여도와 결과 정리

### 셀 23. XGBoost의 중요도 네 가지와 순열 중요도

In [ ]:
from sklearn.inspection import permutation_importance

booster = xgb_final[-1].get_booster()
imp = pd.DataFrame({t: pd.Series(booster.get_score(importance_type=t))
                    for t in ["weight", "gain", "cover", "total_gain"]})
imp = imp.reindex(Z_train.columns).fillna(0)  # 쓰이지 않은 열은 0
perm = permutation_importance(xgb_final, X_test, y_test,
                              scoring="roc_auc", n_repeats=10,
                              random_state=2026)
perm = pd.Series(perm.importances_mean, index=X_test.columns)
print("features used:", (imp["weight"] > 0).sum(), "of", len(imp))
show = ["age", "n_drugs", "bmi", "cost_2022", "insulin", "n_ed", "ckd",
        "hf", "dyslip", "oa", "n_prescribers"]
out = imp.loc[show].round(1)
out["rank_weight"] = imp["weight"].rank(ascending=False,
                                        method="min").loc[show]
out["rank_gain"] = imp["gain"].rank(ascending=False,
                                    method="min").loc[show]
out["perm"] = perm.loc[show].round(4)
out["rank_perm"] = perm.rank(ascending=False, method="min").loc[show]
out.astype({c: int for c in ["rank_weight", "rank_gain", "rank_perm"]})

### 셀 24. SHAP 값을 XGBoost 자체 기능으로 (pred_contribs)

In [ ]:
dtest = xgb.DMatrix(Z_test)
contrib = booster.predict(dtest, pred_contribs=True)  # (3750, 48 + 1)
margin = booster.predict(dtest, output_margin=True)   # 로그 오즈
print(contrib.shape, " largest |sum - log odds|:",
      np.abs(contrib.sum(axis=1) - margin).max().round(6))
shap_df = pd.DataFrame(contrib[:, :-1], columns=Z_test.columns,
                       index=Z_test.index).astype(float)
print("base value:", contrib[0, -1].round(3))

mean_abs = shap_df.abs().mean().sort_values(ascending=False)
top = mean_abs.index[:12][::-1]                    # 위 12개, 아래부터
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4),
                       gridspec_kw={"width_ratios": [1, 1.4]})
ax[0].barh(top, mean_abs[top])
ax[0].set_xlabel("Mean |SHAP| (log odds)")
rng = np.random.default_rng(2026)
for k, f in enumerate(top):
    v = Z_test[f]
    col = v.rank(pct=True)                         # 값의 상대적 크기 0-1
    ok = v.notna().to_numpy()
    yy = k + rng.uniform(-0.3, 0.3, len(v))
    ax[1].scatter(shap_df[f][~ok], yy[~ok], s=4, color="lightgray")
    sc = ax[1].scatter(shap_df[f][ok], yy[ok], s=4, c=col[ok],
                       cmap="coolwarm", vmin=0, vmax=1)
ax[1].set_yticks(range(len(top)), top)
ax[1].axvline(0, color="gray", lw=0.8)
ax[1].set_xlabel("SHAP value (log odds); gray = missing")
fig.colorbar(sc, ax=ax[1], label="Feature value (low to high)")
plt.tight_layout()

### 셀 25. 환자 한 명의 예측을 변수별 기여로 쪼개기

In [ ]:
who = int(np.argsort(-p_xgb)[75])            # 예측 위험이 76번째로 높은 사람
row = shap_df.iloc[who]
base = contrib[who, -1]
big = row[row.abs() >= 0.05].sort_values(key=abs, ascending=False)
print(X_test.iloc[who][["age", "n_ed", "n_drugs", "bmi", "insulin",
                        "hf", "ckd"]].to_dict())
print("base value (log odds):", round(base, 3),
      " = probability", round(1 / (1 + np.exp(-base)), 3))
print(big.round(3).to_string())
print("all other features   :", round(row.sum() - big.sum(), 3))
total = base + row.sum()
print("total (log odds):", round(total, 3), " = probability",
      round(1 / (1 + np.exp(-total)), 3), " | predict_proba:",
      round(p_xgb[who], 3))

fig, ax = plt.subplots(figsize=(6.5, 3.4))
vals = list(big.values) + [row.sum() - big.sum()]
labs = [f"{f} = {Z_test.iloc[who][f]}" for f in big.index] + ["others"]
ax.barh(labs[::-1], vals[::-1],
        color=["tab:red" if v > 0 else "tab:blue" for v in vals[::-1]])
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("SHAP value (log odds)")
plt.tight_layout()

### 셀 26. SHAP 의존 그림과 무리별 평균 (참 구조와 견주기)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(12, 3.2), sharey=True)
for a, f in zip(ax, ["age", "bmi", "n_ed", "n_drugs"]):
    a.scatter(Z_test[f], shap_df[f], s=5, alpha=0.4)
    a.axhline(0, color="gray", lw=0.8)
    a.set_xlabel(f)
ax[0].set_ylabel("SHAP value (log odds)")
plt.tight_layout()

def by(f, groups):
    """특성 f의 SHAP 평균을 groups(사람별 무리 이름)로 묶어서"""
    return shap_df[f].groupby(groups).mean().round(2).to_dict()

age = pd.cut(X_test["age"], [39, 69, 74, 79, 84, 99])
print("age    :", by("age", age.astype(str)))
bmi = pd.cut(X_test["bmi"], [0, 18.5, 23, 25, 30, 60]).astype(str)
bmi[X_test["bmi"].isna()] = "missing"       # 검진을 받지 않은 사람
print("bmi    :", by("bmi", bmi))
print("n_ed   :", by("n_ed", X_test["n_ed"].clip(upper=2)))
print("n_drugs:", by("n_drugs", X_test["n_drugs"] >= 10))
both = X_test["hf"].astype(str) + "/" + X_test["ckd"].astype(str)
print("hf (hf/ckd) :", by("hf", both))
print("ckd (hf/ckd):", by("ckd", both))

### 셀 27. 0–3장 모형 비교 표와 ROC 곡선

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve

models = {
    "logistic (ch 0)": make_pipe(
        LogisticRegression(C=0.1, max_iter=1000)),
    "tree (ch 1)": make_tree_pipe(DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=300, random_state=2026)),
    "random forest (ch 2)": make_tree_pipe(RandomForestClassifier(
        n_estimators=500, max_features=20, min_samples_leaf=85,
        max_samples=0.7339919539656937, random_state=2026, n_jobs=-1)),
    "sklearn GBM": make_tree_pipe(
        GradientBoostingClassifier(random_state=2026)),
    "HistGB tuned": make_hgb_pipe(HistGradientBoostingClassifier(
        categorical_features=[0, 1, 2, 3], random_state=2026,
        **settings["tuned"])),
    "XGBoost default": make_nan_pipe(XGBClassifier(random_state=2026)),
    "XGBoost tuned": xgb_final}
table, boots = [], {}
fig = plt.figure(figsize=(5.2, 4.8))
for name, m in models.items():
    cv = cross_val_score(m, X_train, y_train, cv=skf, scoring="roc_auc")
    p = m.fit(X_train, y_train).predict_proba(X_test)[:, 1]
    boots[name] = boot_auc(y_test, p)
    lo, hi = np.percentile(boots[name], [2.5, 97.5])
    table.append([name, cv.mean(), roc_auc_score(y_test, p), lo, hi])
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, lw=1.2, label=name)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("1 - specificity")
plt.ylabel("Sensitivity")
plt.legend(fontsize=7)
for ref in ["logistic (ch 0)", "random forest (ch 2)"]:
    d = boots["XGBoost tuned"] - boots[ref]
    print(f"XGBoost tuned - {ref}: 95% CI",
          np.percentile(d, [2.5, 97.5]).round(3))
pd.DataFrame(table, columns=["model", "CV AUC", "test AUC", "CI low",
                             "CI high"]).round(3)

### 셀 28. 모형 저장과 불러오기 (save_model과 joblib)

In [ ]:
import os
import joblib

xgb_final[-1].save_model("xgb_admit_2023.json")  # XGBoost 형식: 나무만
joblib.dump(xgb_final, "xgb_admit_2023.joblib")  # 전처리까지 통째로

only_trees = XGBClassifier()
only_trees.load_model("xgb_admit_2023.json")
p1 = only_trees.predict_proba(xgb_final[:-1].transform(X_test))[:, 1]
p2 = joblib.load("xgb_admit_2023.joblib").predict_proba(X_test)[:, 1]
print("max difference:", np.abs(p1 - p_xgb).max(),
      np.abs(p2 - p_xgb).max())
print({f: f"{os.path.getsize(f) / 1024:.0f} KB"
       for f in ["xgb_admit_2023.json", "xgb_admit_2023.joblib"]})

### 셀 29. shap 패키지로 같은 그림

In [ ]:
import shap
print("shap", shap.__version__)

explainer = shap.TreeExplainer(xgb_final[-1])
sv = explainer(Z_test)               # values, base_values, data
print(sv.values.shape, " largest difference from pred_contribs:",
      np.abs(sv.values - contrib[:, :-1]).max())
shap.plots.bar(sv, max_display=12)
shap.plots.beeswarm(sv, max_display=12)
shap.plots.waterfall(sv[who], max_display=10)

### 셀 30. LightGBM과 CatBoost의 시험 AUC (비교 표에 더할 줄)

In [ ]:
lgb_final = make_nan_pipe(LGBMClassifier(
    subsample_freq=1, random_state=2026, verbose=-1,
    **study_lgb.best_params)).fit(X_train, y_train)
cat_final = CatBoostClassifier(cat_features=cat_cols, random_seed=2026,
                               verbose=0).fit(Xk_train, y_train)
b_xgb = boot_auc(y_test, p_xgb)
rows = []
for name, p, cv in [
        ("LightGBM tuned", lgb_final.predict_proba(X_test)[:, 1],
         study_lgb.best_value),
        ("CatBoost default", cat_final.predict_proba(Xk_test)[:, 1],
         cv_cat_default)]:
    b = boot_auc(y_test, p)
    lo, hi = np.percentile(b, [2.5, 97.5])
    d_lo, d_hi = np.percentile(b - b_xgb, [2.5, 97.5])
    rows.append([name, cv, roc_auc_score(y_test, p), lo, hi, d_lo, d_hi])
pd.DataFrame(rows, columns=["model", "CV AUC", "test AUC", "CI low",
                            "CI high", "vs XGB low", "vs XGB high"]
             ).round(3)

## 과제 정답

**과제 1.** 학습률 0.3, 0.1, 0.03에서 조기 종료가 고른 나무 수와 검증 손실, 검증 AUC를 비교합니다.

### 셀 31. 과제 1 정답. 학습률과 조기 종료

In [ ]:
for lr in [0.3, 0.1, 0.03]:
    m = XGBClassifier(n_estimators=5000, learning_rate=lr, max_depth=2,
                      early_stopping_rounds=50, eval_metric="logloss",
                      random_state=2026)
    m.fit(Z_tr, y_tr, eval_set=[(Z_val, y_val)], verbose=False)
    auc = roc_auc_score(y_val, m.predict_proba(Z_val)[:, 1])
    print(f"learning_rate {lr}: trees {m.best_iteration + 1},"
          f" log loss {m.best_score:.4f}, validation AUC {auc:.4f}")

**과제 2.** scale_pos_weight를 1과 (음성 수 ÷ 양성 수)로 바꿔 겹 밖 예측의 AUC, 평균 확률, 혼동행렬을 비교합니다.

### 셀 32. 과제 2 정답. scale_pos_weight와 확률, 혼동행렬

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

spw = (y_train == 0).sum() / (y_train == 1).sum()   # 10791 / 459
for w in [1, spw]:
    m = make_nan_pipe(XGBClassifier(random_state=2026,
                                    scale_pos_weight=w, **best_xgb))
    p = cross_val_predict(m, X_train, y_train, cv=skf,
                          method="predict_proba")[:, 1]
    print(f"scale_pos_weight={w:.1f}: AUC {roc_auc_score(y_train, p):.4f},"
          f" mean prob {p.mean():.3f}")
    print(confusion_matrix(y_train, (p >= 0.5).astype(int)))

**과제 3.** 히스토그램 부스팅을 Optuna로 30번 튜닝합니다.

### 셀 33. 과제 3 정답. 히스토그램 부스팅을 Optuna로

In [ ]:
def hgb_objective(trial):
    model = make_hgb_pipe(HistGradientBoostingClassifier(
        categorical_features=[0, 1, 2, 3], early_stopping=False,
        random_state=2026,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3,
                                          log=True),
        max_iter=trial.suggest_int("max_iter", 50, 1000, step=50),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 2, 32,
                                         log=True),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 200,
                                           log=True),
        l2_regularization=trial.suggest_float("l2_regularization",
                                              0.001, 10, log=True)))
    return cross_val_score(model, X_train, y_train, cv=skf,
                           scoring="roc_auc").mean()

study_hgb = optuna.create_study(direction="maximize",
                                sampler=TPESampler(seed=2026))
study_hgb.optimize(hgb_objective, n_trials=30)
print({k: round(v, 4) for k, v in study_hgb.best_params.items()})
print(round(study_hgb.best_value, 4), " trial",
      study_hgb.best_trial.number)